# Darknode GPT — train a cyber-leaning language model **from scratch**

A nanoGPT-style decoder transformer trained from zero (no fine-tuning) on
**Karpathy's FineWeb-Edu** corpus mixed with the **Primus** cybersecurity datasets
(Trend Micro AI Lab). Tokenizer: GPT-2 BPE (tiktoken). Architecture: GPT-2 family.

### Honest expectation
Training from scratch on Kaggle's free GPUs realistically yields a **small, GPT-2-class
model (~124M params)**. It will generate fluent text and absorb cybersecurity vocabulary,
but it is **not** an instruction-following assistant and won't match a fine-tuned 8B model.
Reaching real GPT-2 quality (~10B tokens) takes **several Kaggle sessions** — this notebook
checkpoints to your Hugging Face repo and **resumes automatically** so you can accumulate
training across runs.

### Setup before Run All
1. **Settings → Accelerator →** GPU `T4 x2` (or `P100`).
2. **Settings → Internet → On** (needed to stream the datasets + push to the Hub).
3. **Add-ons → Secrets →** add a secret named `HF_TOKEN` with a Hugging Face **write** token
   (from https://huggingface.co/settings/tokens). Uploads go to **`Darknod/Darknode`**.
4. Run all cells. First run builds the token shards (~20-40 min) then trains until the
   wall-clock limit, saving/pushing a checkpoint. Re-run in a new session to continue.

In [ ]:
!pip -q install -U tiktoken datasets huggingface_hub

## 1. Configuration
`model_size`: `nano` (~30M, fast) · `small` (~124M, GPT-2 small) · `medium` (~350M, slow).
Start with `small`; drop to `nano` for a quick end-to-end smoke test. Bump
`target_train_tokens` once you're happy it runs.

In [ ]:
CONFIG = dict(
    hf_repo="Darknod/Darknode",   # where checkpoints + the final model are pushed
    push_to_hub=True,              # False => train locally only (no upload / no cross-session resume)
    push_every=1000,               # upload a checkpoint to the Hub every N steps
    resume=True,                   # resume from a local or Hub checkpoint if one exists

    model_size="small",           # nano | small | medium
    block_size=1024,               # context length
    cyber_ratio=0.2,               # fraction of training docs drawn from the Primus cyber corpora

    target_train_tokens=300_000_000,  # tokens to tokenize/train on (raise to grow the model)
    val_tokens=5_000_000,

    micro_batch=4,                 # per-step micro-batch (lower to 2 if you hit CUDA OOM)
    grad_accum=8,                  # effective batch = micro_batch * block_size * grad_accum tokens
    max_steps=0,                   # 0 => derive from target_train_tokens

    eval_every=250, sample_every=500, save_every=500,
    max_lr=6e-4, min_lr=6e-5, warmup_steps=150, weight_decay=0.1, grad_clip=1.0,
    wall_clock_hours=8.5,          # checkpoint + stop before Kaggle kills the session

    data_dir="/kaggle/working/data", out_dir="/kaggle/working", seed=1337,
)

## 2. Environment, precision, HF token

In [ ]:
import os, math, time, random, json, inspect
import numpy as np, torch, torch.nn as nn
from torch.nn import functional as F
from types import SimpleNamespace
from contextlib import nullcontext

torch.manual_seed(CONFIG["seed"]); random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()   # T4 => fp16, A100+ => bf16
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=amp_dtype) if device == "cuda" else nullcontext()

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("device:", device, "| amp:", amp_dtype, "| HF token:", "yes" if HF_TOKEN else "no")
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Data — stream FineWeb-Edu + Primus, tokenize to binary shards
Writes `train.bin` / `val.bin` (uint16 GPT-2 token ids). The validation split is drawn
first and kept disjoint from training. A missing/blocked dataset is skipped gracefully.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
EOT = enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

def encode_doc(text):
    ids = enc.encode_ordinary(text); ids.append(EOT); return ids

from datasets import load_dataset
def text_of(ex):
    for k in ("text", "content", "raw", "markdown", "body", "instruction"):
        v = ex.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return " ".join(str(v) for v in ex.values() if isinstance(v, str))

def fineweb_stream():
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    for ex in ds:
        t = text_of(ex)
        if t: yield t

def cyber_stream():
    for repo in ("trendmicro-ailab/Primus-Seed", "trendmicro-ailab/Primus-FineWeb"):
        try:
            ds = load_dataset(repo, split="train", streaming=True)
        except Exception as e:
            print("  skip", repo, "->", repr(e)[:120]); continue
        for ex in ds:
            t = text_of(ex)
            if t: yield t

def _ntok(path):
    return os.path.getsize(path) // 2 if os.path.exists(path) else 0

def build_dataset():
    os.makedirs(CONFIG["data_dir"], exist_ok=True)
    train_bin = os.path.join(CONFIG["data_dir"], "train.bin")
    val_bin   = os.path.join(CONFIG["data_dir"], "val.bin")
    need_t, need_v = CONFIG["target_train_tokens"], CONFIG["val_tokens"]
    if _ntok(train_bin) >= need_t and _ntok(val_bin) >= need_v:
        print("data ready:", _ntok(train_bin), "train /", _ntok(val_bin), "val tokens"); return train_bin, val_bin
    print("building token shards (streaming FineWeb-Edu + Primus) ...")
    fw, cy = fineweb_stream(), cyber_stream()
    dump = lambda f, ids: np.array(ids, dtype=np.uint16).tofile(f)

    vw = 0
    with open(val_bin, "wb") as f:
        while vw < need_v:
            ids = encode_doc(next(fw)); dump(f, ids); vw += len(ids)

    tw = 0; cyber_done = False; t0 = time.time(); mark = 0
    with open(train_bin, "wb") as f:
        while tw < need_t:
            take_cyber = (not cyber_done) and (random.random() < CONFIG["cyber_ratio"])
            if take_cyber:
                try: doc = next(cy)
                except StopIteration: cyber_done = True; doc = next(fw)
            else:
                doc = next(fw)
            ids = encode_doc(doc); dump(f, ids); tw += len(ids)
            if tw - mark >= 20_000_000:
                mark = tw; rate = tw / max(1e-6, time.time() - t0) / 1e3
                print("  {:.0f}M train tokens  ({:.0f}k tok/s){}".format(tw/1e6, rate, "  [cyber exhausted]" if cyber_done else ""))
    print("done:", _ntok(train_bin), "train /", _ntok(val_bin), "val tokens")
    return train_bin, val_bin

train_bin, val_bin = build_dataset()

## 4. Model — nanoGPT-style GPT (flash attention, weight tying)

In [ ]:
PRESETS = {
    "nano":   dict(n_layer=6,  n_head=6,  n_embd=384),
    "small":  dict(n_layer=12, n_head=12, n_embd=768),
    "medium": dict(n_layer=24, n_head=16, n_embd=1024),
}

class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.c_attn = nn.Linear(c.n_embd, 3 * c.n_embd, bias=c.bias)
        self.c_proj = nn.Linear(c.n_embd, c.n_embd, bias=c.bias); self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head, self.n_embd, self.dropout = c.n_head, c.n_embd, c.dropout
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_fc = nn.Linear(c.n_embd, 4 * c.n_embd, bias=c.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * c.n_embd, c.n_embd, bias=c.bias); self.c_proj.NANOGPT_SCALE_INIT = 1
        self.drop = nn.Dropout(c.dropout)
    def forward(self, x): return self.drop(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, bias=c.bias); self.attn = CausalSelfAttention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, bias=c.bias); self.mlp = MLP(c)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x)); x = x + self.mlp(self.ln_2(x)); return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__(); self.cfg = c
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(c.vocab_size, c.n_embd), wpe=nn.Embedding(c.block_size, c.n_embd),
            drop=nn.Dropout(c.dropout), h=nn.ModuleList([Block(c) for _ in range(c.n_layer)]),
            ln_f=nn.LayerNorm(c.n_embd, bias=c.bias)))
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   # weight tying
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02
            if hasattr(m, "NANOGPT_SCALE_INIT"): std *= (2 * self.cfg.n_layer) ** -0.5
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for b in self.transformer.h: x = b(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return self.lm_head(x[:, [-1], :]), None
    def configure_optimizers(self, wd, lr, dev):
        ps = [p for p in self.parameters() if p.requires_grad]
        groups = [{"params": [p for p in ps if p.dim() >= 2], "weight_decay": wd},
                  {"params": [p for p in ps if p.dim() < 2], "weight_decay": 0.0}]
        fused = (dev == "cuda") and ("fused" in inspect.signature(torch.optim.AdamW).parameters)
        return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), eps=1e-8, **({"fused": True} if fused else {}))

## 5. Batch loader (memory-mapped shards)

In [ ]:
class Loader:
    def __init__(self, path, B, T):
        self.data = np.memmap(path, dtype=np.uint16, mode="r"); self.B, self.T = B, T
        self.n = len(self.data); self.pos = 0
    def next(self):
        B, T = self.B, self.T; need = B * T + 1
        if self.pos + need > self.n: self.pos = 0
        buf = torch.from_numpy(self.data[self.pos:self.pos + need].astype(np.int64))
        self.pos += B * T
        x = buf[:-1].view(B, T); y = buf[1:].view(B, T)
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

## 6. Train (with eval, sampling, checkpoint + cross-session resume via the Hub)

In [ ]:
cfg = SimpleNamespace(block_size=CONFIG["block_size"], vocab_size=50304, dropout=0.0, bias=False, **PRESETS[CONFIG["model_size"]])
model = GPT(cfg).to(device)
print("model {}: {:.1f}M params".format(CONFIG["model_size"], sum(p.numel() for p in model.parameters()) / 1e6))
opt = model.configure_optimizers(CONFIG["weight_decay"], CONFIG["max_lr"], device)
scaler = (torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16)) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16)))
train_loader, val_loader = Loader(train_bin, CONFIG["micro_batch"], CONFIG["block_size"]), Loader(val_bin, CONFIG["micro_batch"], CONFIG["block_size"])

tokens_per_step = CONFIG["micro_batch"] * CONFIG["block_size"] * CONFIG["grad_accum"]
max_steps = CONFIG["max_steps"] or max(1, CONFIG["target_train_tokens"] // tokens_per_step)
ckpt_path = os.path.join(CONFIG["out_dir"], "ckpt.pt")
print("tokens/step:", tokens_per_step, "| max_steps:", max_steps)

start_step = 0
if CONFIG["resume"]:
    p = ckpt_path
    if not os.path.exists(p) and CONFIG["push_to_hub"] and HF_TOKEN:
        try:
            from huggingface_hub import hf_hub_download
            p = hf_hub_download(CONFIG["hf_repo"], "ckpt.pt", token=HF_TOKEN)
        except Exception as e:
            print("no Hub checkpoint:", repr(e)[:120]); p = None
    if p and os.path.exists(p):
        ck = torch.load(p, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start_step = ck["step"]
        print("resumed from step", start_step)

def get_lr(it):
    wu = CONFIG["warmup_steps"]
    if it < wu: return CONFIG["max_lr"] * (it + 1) / wu
    if it > max_steps: return CONFIG["min_lr"]
    coeff = 0.5 * (1 + math.cos(math.pi * (it - wu) / max(1, max_steps - wu)))
    return CONFIG["min_lr"] + coeff * (CONFIG["max_lr"] - CONFIG["min_lr"])

@torch.no_grad()
def eval_val(iters=50):
    model.eval(); losses = []
    for _ in range(iters):
        x, y = val_loader.next()
        with amp_ctx(): _, loss = model(x, y)
        losses.append(loss.item())
    model.train(); return sum(losses) / len(losses)

def save_ckpt(step, push=False):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step, "cfg": vars(cfg)}, ckpt_path)
    if push and CONFIG["push_to_hub"] and HF_TOKEN:
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=HF_TOKEN); api.create_repo(CONFIG["hf_repo"], exist_ok=True)
            api.upload_file(path_or_fileobj=ckpt_path, path_in_repo="ckpt.pt", repo_id=CONFIG["hf_repo"])
            print("  pushed ckpt @ step", step, "->", CONFIG["hf_repo"])
        except Exception as e:
            print("  push failed:", repr(e)[:160])

model.train(); t_start = time.time(); wall = CONFIG["wall_clock_hours"] * 3600; step = start_step
for step in range(start_step, max_steps):
    for g in opt.param_groups: g["lr"] = get_lr(step)
    opt.zero_grad(set_to_none=True); loss_acc = 0.0
    for _ in range(CONFIG["grad_accum"]):
        x, y = train_loader.next()
        with amp_ctx(): _, loss = model(x, y)
        loss = loss / CONFIG["grad_accum"]; loss_acc += loss.item()
        scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(opt); scaler.update()
    if step % 20 == 0:
        print("step {}/{}  loss {:.4f}  lr {:.2e}  {:.1f}m".format(step, max_steps, loss_acc, get_lr(step), (time.time() - t_start) / 60))
    if step > start_step and step % CONFIG["eval_every"] == 0:
        print("  val loss {:.4f}".format(eval_val()))
    if step > start_step and step % CONFIG["save_every"] == 0:
        save_ckpt(step, push=(step % CONFIG["push_every"] == 0))
    if time.time() - t_start > wall:
        print("wall-clock limit reached — checkpoint + stop"); save_ckpt(step, push=True); break
save_ckpt(step + 1, push=True)
print("paused/finished at step", step + 1, "of", max_steps)

## 7. Sample from the model

In [ ]:
@torch.no_grad()
def generate(prompt, max_new=120, temperature=0.8, top_k=50):
    model.eval()
    x = torch.tensor(enc.encode_ordinary(prompt), dtype=torch.long, device=device)[None, ...]
    for _ in range(max_new):
        logits, _ = model(x[:, -CONFIG["block_size"]:])
        logits = logits[:, -1, :] / temperature
        if top_k:
            v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        x = torch.cat((x, torch.multinomial(probs, 1)), dim=1)
    model.train(); return enc.decode(x[0].tolist())

for p in ["The attacker exploited a", "A buffer overflow occurs when", "To investigate a phishing email,"]:
    print("---", p); print(generate(p)); print()

## 8. Save the final model and push to `Darknod/Darknode`
Produces a clean folder (`pytorch_model.bin` + `config.json` + `README.md`) and uploads it.
Equivalent CLI (run locally after downloading the folder):
```
curl -LsSf https://hf.co/cli/install.sh | bash
hf auth login
hf upload Darknod/Darknode ./darknode-model
```

In [ ]:
final_dir = os.path.join(CONFIG["out_dir"], "darknode-model"); os.makedirs(final_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(final_dir, "pytorch_model.bin"))
json.dump({**vars(cfg), "tokenizer": "gpt2 (tiktoken)", "arch": "nanoGPT-style decoder transformer",
           "trained_on": "FineWeb-Edu + Primus (Trend Micro) cyber corpora"},
          open(os.path.join(final_dir, "config.json"), "w"), indent=2)
open(os.path.join(final_dir, "README.md"), "w").write(
    "# Darknode GPT\n\nA GPT-2-family language model trained **from scratch** on FineWeb-Edu "
    "mixed with the Primus cybersecurity datasets. Tokenizer: GPT-2 BPE (tiktoken). "
    "Load with the GPT class from the training notebook.\n")

if CONFIG["push_to_hub"] and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN); api.create_repo(CONFIG["hf_repo"], exist_ok=True)
    api.upload_folder(folder_path=final_dir, repo_id=CONFIG["hf_repo"])
    print("uploaded -> https://huggingface.co/" + CONFIG["hf_repo"])
else:
    print("push disabled or no HF_TOKEN — local model saved in", final_dir)

## Continuing & scaling
- **More training:** just re-run this notebook in a fresh session — it pulls the latest
  `ckpt.pt` from `Darknod/Darknode` and resumes. Each session adds ~8.5h of training.
- **Bigger model:** set `model_size="medium"` and lower `micro_batch` to 2 if you hit OOM.
- **More data:** raise `target_train_tokens` (and delete `/kaggle/working/data` to rebuild).
- **Want raw capability instead of from-scratch?** A QLoRA fine-tune of Llama-3.1-8B-Instruct
  on this same cyber data produces a far stronger assistant — ask for that notebook.